In [ ]:
import pandas as pd
from scipy.stats import ttest_ind

DATA = '../data/case-study/processed'
PSY = '../data/case-study/psychometric'

# load data
hr_bl01 = pd.read_csv(f'{DATA}/hr.csv')
hr_tk01 = pd.read_csv(f'{DATA}/hr_01.csv')

# high confidence only
baseline_heart_rate = hr_bl01[hr_bl01['confidence'] == 1.0]['heart_rate']
test_heart_rate = hr_tk01[hr_tk01['confidence'] == 1.0]['heart_rate']

# calculate averages
avg_hr_baseline = baseline_heart_rate.mean()
avg_hr_test = test_heart_rate.mean()

# Welch's t-test
t_stat, p_value = ttest_ind(baseline_heart_rate, test_heart_rate, equal_var=False)

# percentage increase
percentage_increase = ((avg_hr_test - avg_hr_baseline) / avg_hr_baseline) * 100

results = {
    'Average Heart Rate - Baseline': avg_hr_baseline,
    'Average Heart Rate - Test': avg_hr_test,
    "Welch's T-Test Statistic": t_stat,
    "Welch's T-Test P-Value": p_value,
    'Percentage Increase in Heart Rate': percentage_increase
}

results

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

DATA = '../data/case-study/processed'

# load and filter
hr_baseline = pd.read_csv(f'{DATA}/hr.csv')
hr_01 = pd.read_csv(f'{DATA}/hr_01.csv')
hr_02 = pd.read_csv(f'{DATA}/hr_02.csv')
hr_03 = pd.read_csv(f'{DATA}/hr_03.csv')

baseline = hr_baseline[hr_baseline['confidence'] == 1.0]['heart_rate']
test_01 = hr_01[hr_01['confidence'] == 1.0]['heart_rate']
test_02 = hr_02[hr_02['confidence'] == 1.0]['heart_rate']
test_03 = hr_03[hr_03['confidence'] == 1.0]['heart_rate']

sessions = {'Baseline': baseline, 'Session 01': test_01, 'Session 02': test_02, 'Session 03': test_03}

# normality tests
print("=== Shapiro-Wilk Normality Tests ===\n")
for name, data in sessions.items():
    sample = data.sample(min(5000, len(data)), random_state=42)
    stat, p = stats.shapiro(sample)
    normal = "normal" if p > 0.05 else "non-normal"
    print(f"{name}: W={stat:.4f}, p={p:.4f} ({normal}), n={len(data)}")

# Cohen's d
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled = np.sqrt(((nx-1)*x.std()**2 + (ny-1)*y.std()**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled if pooled > 0 else 0.0

# paired comparisons vs baseline
print("\n=== Baseline vs Session Comparisons ===\n")
for name, data in list(sessions.items())[1:]:
    t_stat, p_val = stats.ttest_ind(baseline, data, equal_var=False)
    d = cohens_d(data, baseline)
    size = "large" if abs(d) >= 0.8 else "medium" if abs(d) >= 0.5 else "small"

    # 95% CI on mean difference
    diff = data.mean() - baseline.mean()
    se = np.sqrt(data.std()**2/len(data) + baseline.std()**2/len(baseline))
    ci = (diff - 1.96*se, diff + 1.96*se)

    print(f"{name} vs Baseline:")
    print(f"  Mean diff: {diff:.2f} BPM, 95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"  Welch's t={t_stat:.3f}, p={p_val:.4f}")
    print(f"  Cohen's d={d:.3f} ({size})")
    print()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/case-study/processed'
PSY = '../data/case-study/psychometric'

sessions = [1, 2, 3]
hr_data = {}
psychometric = {}

for s in sessions:
    hr = pd.read_csv(f'{DATA}/hr_{s:02d}.csv')
    hr = hr[hr['confidence'] == 1.0]
    hr['datetime'] = pd.to_datetime(hr['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    hr_data[s] = hr.sort_values('datetime')

    psy = pd.read_csv(f'{PSY}/Psychometric_Test_Results_{s:02d}.csv')
    psy['Question Start Time'] = pd.to_datetime(psy['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    psychometric[s] = psy

question_types = ['HADS', 'STAI-T', 'STAI-S', 'BFI', 'FQ']

def process_and_plot(type_key):
    plt.figure(figsize=(14, 7))

    for s in sessions:
        data = psychometric[s][psychometric[s]['Type'] == type_key].reset_index(drop=True)
        hr = hr_data[s]

        merged = pd.merge_asof(
            data.sort_values('Question Start Time'),
            hr[['datetime', 'heart_rate']],
            left_on='Question Start Time',
            right_on='datetime',
            direction='nearest'
        )

        extracted = merged['Test'].str.extract(r'(\d+)')
        merged['Question Number'] = extracted[0].fillna(0).astype(int)
        merged['Time Difference'] = (merged['Question Start Time'] - merged['Question Start Time'].min()).dt.total_seconds()

        plt.plot(merged['Time Difference'], merged['heart_rate'], label=f'HR - {type_key} Session {s}', marker='o')

        for i in range(len(merged)):
            plt.annotate(int(merged['Question Number'].iloc[i]),
                         (merged['Time Difference'].iloc[i], merged['heart_rate'].iloc[i]))

    plt.title(f'Heart Rate During {type_key} Questions')
    plt.xlabel('Time Difference (seconds)')
    plt.ylabel('Heart Rate')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

for q_type in question_types:
    process_and_plot(q_type)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DATA = '../data/case-study/processed'
PSY = '../data/case-study/psychometric'

categories = {'HADS': 14, 'STAI-S': 20, 'STAI-T': 20, 'BFI': 10, 'FQ': 24}

for session_num in [1, 2, 3]:
    hr_data = pd.read_csv(f'{DATA}/hr_{session_num:02d}.csv', delimiter=',')
    hr_data = hr_data[hr_data['confidence'] == 1.0]
    hr_data['datetime'] = pd.to_datetime(hr_data['datetime'], format='%Y/%m/%d %H:%M:%S.%f', utc=True, errors='coerce').dt.tz_convert(None)

    psychometric_data = pd.read_csv(f'{PSY}/Psychometric_Test_Results_{session_num:02d}.csv')
    
    for category_name, expected_questions in categories.items():
        category_data = psychometric_data[psychometric_data['Type'] == category_name].copy()
        category_data['Question Start Time'] = pd.to_datetime(category_data['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
        category_data['Question Answer Time'] = pd.to_datetime(category_data['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

        # accumulate segments
        segments = []
        question_times = []

        for _, row in category_data.iterrows():
            start_time = row['Question Start Time']
            end_time = row['Question Answer Time']
            mask = (hr_data['datetime'] >= start_time) & (hr_data['datetime'] <= end_time)
            seg = hr_data.loc[mask]
            if not seg.empty:
                segments.append(seg)
            question_times.append(end_time)

        category_hr_data = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()

        if len(question_times) != expected_questions:
            print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

        category_hr_data = category_hr_data.sort_values(by='datetime').reset_index(drop=True)

        window_size = 10
        category_hr_data['smoothed_heart_rate'] = category_hr_data['heart_rate'].rolling(window=window_size).mean()

        plt.figure(figsize=(12, 6))
        plt.plot(category_hr_data['datetime'], category_hr_data['smoothed_heart_rate'], label='Heart Rate', color='b')

        for i, time in enumerate(question_times, start=1):
            nearest_idx = category_hr_data['datetime'].searchsorted(time)
            if nearest_idx < len(category_hr_data):
                hr_at_time = category_hr_data.iloc[nearest_idx]['smoothed_heart_rate']
                plt.scatter(time, hr_at_time, color='red')
                plt.text(time, hr_at_time + 1, f'Q{i}', rotation=45, ha='right')
            else:
                plt.axvline(x=time, color='red', linestyle='--', alpha=0.5)
                plt.text(time, category_hr_data['smoothed_heart_rate'].max(), f'Q{i}', rotation=45, ha='right')

        plt.xlabel('Time')
        plt.ylabel('Heart Rate (bpm)')
        plt.title(f'Heart Rate Changes During {category_name} (Session {session_num})')
        plt.xticks(rotation=45)
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        plt.close()